In [1]:
import pandas as pd
import numpy as np

In [2]:
nav_df = pd.read_csv("data/raw/02_nav_history.csv")
nav_df.head()


,amfi_code,date,nav
0,119551,2022-01-03,54.3856
1,119551,2022-01-04,54.3474
2,119551,2022-01-05,54.6869
3,119551,2022-01-06,55.4550
4,119551,2022-01-07,55.3692


In [3]:
nav_df["date"] = pd.to_datetime(nav_df["date"])
nav_df = nav_df.sort_values(["amfi_code", "date"])

nav_df.head()

,amfi_code,date,nav
5750,100016,2022-01-03,520.4608
5751,100016,2022-01-04,515.0971
5752,100016,2022-01-05,521.7239
5753,100016,2022-01-06,515.7880
5754,100016,2022-01-07,515.1639


In [4]:
nav_df["return"] = nav_df.groupby("amfi_code")["nav"].pct_change()

nav_df.head()

,amfi_code,date,nav,return
5750,100016,2022-01-03,520.4608,NaN
5751,100016,2022-01-04,515.0971,-0.010306
5752,100016,2022-01-05,521.7239,0.012865
5753,100016,2022-01-06,515.7880,-0.011377
5754,100016,2022-01-07,515.1639,-0.001210


In [5]:
import numpy as np

df = nav_df.dropna(subset=["return"])

results = []

for fund in df["amfi_code"].unique():
    fund_returns = df[df["amfi_code"] == fund]["return"]
    
    var_95 = np.percentile(fund_returns, 5)
    cvar_95 = fund_returns[fund_returns <= var_95].mean()
    
    results.append([fund, var_95, cvar_95])

var_cvar_df = pd.DataFrame(results, columns=["amfi_code", "VaR_95", "CVaR_95"])

var_cvar_df.head()

,amfi_code,VaR_95,CVaR_95
0,100016,-0.014364,-0.018060
1,100025,-0.003793,-0.004994
2,100033,-0.019034,-0.023456
3,101206,-0.013282,-0.017439
4,101207,-0.026021,-0.032459


In [6]:
var_cvar_df.to_csv("var_cvar_report.csv", index=False)
print("Var-CVaR file saved successfully 🚀")

Var-CVaR file saved successfully 🚀


In [7]:
import numpy as np

top_funds = nav_df["amfi_code"].unique()[:5]

top_funds

array([100016, 100025, 100033, 101206, 101207])

In [8]:
import numpy as np

fund_id = top_funds[0]

fund_data = nav_df[nav_df["amfi_code"] == fund_id].copy()
fund_data = fund_data.dropna(subset=["return"])

fund_data["rolling_mean"] = fund_data["return"].rolling(90).mean()
fund_data["rolling_std"] = fund_data["return"].rolling(90).std()

fund_data["rolling_sharpe"] = (fund_data["rolling_mean"] / fund_data["rolling_std"]) * np.sqrt(252)

fund_data[["date", "rolling_sharpe"]].tail()

,date,rolling_sharpe
6895,2026-05-25,-0.986168
6896,2026-05-26,-0.957388
6897,2026-05-27,-1.244102
6898,2026-05-28,-1.191463
6899,2026-05-29,-1.215738


In [9]:
import numpy as np

results = []

for fund_id in top_funds:
    fund_data = nav_df[nav_df["amfi_code"] == fund_id].copy()
    fund_data = fund_data.dropna(subset=["return"])

    fund_data["rolling_mean"] = fund_data["return"].rolling(90).mean()
    fund_data["rolling_std"] = fund_data["return"].rolling(90).std()

    fund_data["rolling_sharpe"] = (fund_data["rolling_mean"] / fund_data["rolling_std"]) * np.sqrt(252)

    avg_sharpe = fund_data["rolling_sharpe"].mean()

    results.append([fund_id, avg_sharpe])

sharpe_df = pd.DataFrame(results, columns=["amfi_code", "avg_rolling_sharpe"])

sharpe_df

,amfi_code,avg_rolling_sharpe
0,100016,0.337507
1,100025,1.125502
2,100033,1.586154
3,101206,1.484648
4,101207,0.416205


In [10]:
holdings = pd.read_csv("data/raw/09_portfolio_holdings.csv")

holdings["weight_pct"] = pd.to_numeric(holdings["weight_pct"], errors="coerce")

holdings["weight_norm"] = holdings.groupby("amfi_code")["weight_pct"].transform(lambda x: x / x.sum())

hhi_df = holdings.groupby("amfi_code").apply(
    lambda x: (x["weight_norm"] ** 2).sum()
).reset_index()

hhi_df.columns = ["amfi_code", "HHI"]

hhi_df.head()

,amfi_code,HHI
0,100016,0.139534
1,100033,0.147592
2,101206,0.129306
3,101207,0.200741
4,102885,0.174709


In [11]:
print("TOP RISK FUNDS (Lowest VaR):")
print(var_cvar_df.sort_values("VaR_95").head())

print("\nTOP PERFORMING FUNDS (Sharpe):")
print(sharpe_df.sort_values("avg_rolling_sharpe", ascending=False).head())

print("\nMOST CONCENTRATED FUNDS (HHI):")
print(hhi_df.sort_values("HHI", ascending=False).head())

TOP RISK FUNDS (Lowest VaR):
    amfi_code    VaR_95   CVaR_95
22     119599 -0.026859 -0.032384
17     119095 -0.026188 -0.031667
4      101207 -0.026021 -0.032459
11     118634 -0.025438 -0.032304
21     119598 -0.024507 -0.030595

TOP PERFORMING FUNDS (Sharpe):
   amfi_code  avg_rolling_sharpe
2     100033            1.586154
3     101206            1.484648
1     100025            1.125502
4     101207            0.416205
0     100016            0.337507

MOST CONCENTRATED FUNDS (HHI):
    amfi_code       HHI
11     119092  0.206489
3      101207  0.200741
18     119599  0.174751
4      102885  0.174709
7      118632  0.168231


In [12]:
var_cvar_df.to_csv("var_cvar_report.csv", index=False)
sharpe_df.to_csv("sharpe_report.csv", index=False)
hhi_df.to_csv("hhi_report.csv", index=False)

print("🎉 DAY 6 COMPLETED SUCCESSFULLY")

🎉 DAY 6 COMPLETED SUCCESSFULLY
